In [ ]:
# Install packages to make sure everything works
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers",
    "datasets",
    "torch",
    "sentence-transformers",
])

In [ ]:
# Import of dataset from huggingface
from datasets import load_dataset
import datasets

# Suppress info/debug logging, unnecessary output
datasets.logging.set_verbosity_error()

mnlp_dataset = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")
# Extract train and test splits
train_data = mnlp_dataset["train"]
test_data = mnlp_dataset["test"]
blind_data = mnlp_dataset["blind"]

In [ ]:
# Visualize dataset structure
print("Dataset structure:")
print(mnlp_dataset)

print("\nSample data:")
print(f"{mnlp_dataset['train'][0]}")

## Task [B1]  
Implement the two baseline semantic search systems:  
- distilbert-base-uncased
- all-MiniLM-L6-v2

In [ ]:
from sentence_transformers import SentenceTransformer, models

# Load DistilBERT specifically using the HuggingFace model name
# Create Transformer module from the specific DistilBERT model
word_embedding_model = models.Transformer("distilbert-base-uncased")
# Add pooling layer (mean pooling)
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
# Combine into a SentenceTransformer model
model_bert = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# Load MiniLM using SentenceTransformer's built-in support for pre-trained models
model_mini = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Cosine similarity search utility function
def cosine_similarity_search(query_embedding, chunk_embeddings, top_k=3):
    """Find top-k chunks most similar to query using cosine similarity"""
    similarities = cosine_similarity(query_embedding.reshape(1, -1), chunk_embeddings).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return top_indices, similarities[top_indices]

Implementation of Hit@k metric as:  
$$Hit@k = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1} [r_i \leq k]$$

In [ ]:

def hit_at_k(retrieved_indices, ground_truth_indices, k):
    """Hit@k: 1 if any ground truth in top-k results, else 0"""
    top_k_results = retrieved_indices[:k]
    gt_set = set(ground_truth_indices)
    has_hit = any(item in gt_set for item in top_k_results)
    return 1 if has_hit else 0

## [A2]
Report additional performance metrics beyond Hit@k

Implementation of MRR@k metric as:  
$$MRR@k = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{rank_i} \cdot \mathbb{1}[rank_i \leq k]$$

In [ ]:
def mrr_at_k(retrieved_indices, ground_truth_indices, k):
    """MRR@k: Reciprocal rank of first k relevant items"""

    relevant_items = set(ground_truth_indices)
    
    # Consider only first k elements
    top_k_predictions = retrieved_indices[:k]
    
    for rank, item_id in enumerate(top_k_predictions, start=1):
        if item_id in relevant_items:
            # Found a relevant item at this rank, return reciprocal
            return 1.0 / rank
            
    # No relevant item found in top-k
    return 0.0

In [ ]:
# Function to evaluate retrieval using both Hit@k and MRR@k metrics
def evaluate_retrieval(queries_embeddings, chunk_embeddings, ground_truths, k_values=[1, 3, 5]):
    results = {}
    for k in k_values:
        hits = [hit_at_k(cosine_similarity_search(q, chunk_embeddings, k)[0], ground_truths[i], k) 
                for i, q in enumerate(queries_embeddings)]
        mrrs = [mrr_at_k(cosine_similarity_search(q, chunk_embeddings, k)[0], ground_truths[i], k) 
                for i, q in enumerate(queries_embeddings)]
        results[f'Hit@{k}'] = np.mean(hits)
        results[f'MRR@{k}'] = np.mean(mrrs)
    return results

In [ ]:
# Helper function for evaluating on real dataset
def evaluate_model_on_dataset(model, dataset, model_name="Model"):
    """Evaluate model on real dataset using Hit@k and MRR@k metrics"""
    print(f"Evaluating {model_name} on {len(dataset)} test samples...")
    
    hits_per_k = {k: [] for k in [1, 3, 5]}
    mrrs_per_k = {k: [] for k in [1, 3, 5]}
    
    for idx, example in enumerate(dataset):
        query = example['query']
        candidates = example['candidate_chunks']
        answer_pos = example['answer_pos']
        
        # Encode query and candidates
        query_emb = model.encode(query, convert_to_numpy=True)
        candidates_emb = model.encode(candidates, convert_to_numpy=True)
        
        # Compute similarities and ranking
        similarities = cosine_similarity(query_emb.reshape(1, -1), candidates_emb).flatten()
        ranked_indices = np.argsort(similarities)[::-1]
        
        # Use hit_at_k and mrr_at_k functions for evaluation
        for k in [1, 3, 5]:
            hits_per_k[k].append(hit_at_k(ranked_indices, [answer_pos], k))
            mrrs_per_k[k].append(mrr_at_k(ranked_indices, [answer_pos], k))
        
        if (idx + 1) % max(1, len(dataset) // 5) == 0:
            print(f"  Progress: {(idx + 1) / len(dataset) * 100:.1f}%")
    
    # Calculate average metrics
    results = {}
    for k in [1, 3, 5]:
        results[f'Hit@{k}'] = np.mean(hits_per_k[k])
        results[f'MRR@{k}'] = np.mean(mrrs_per_k[k])
    
    print(f"Evaluation completed")
    return results

In [ ]:
# Evaluate Baseline 1: DistilBERT
print("="*70)
print("BASELINE 1: DistilBERT")
print("="*70 + "\n")
bert_results = evaluate_model_on_dataset(model_bert, test_data, "DistilBERT")
print("DistilBERT Results:")
for metric in sorted(bert_results.keys()):
    print(f"  {metric}: {bert_results[metric]:.4f}")

In [ ]:
# Evaluate Baseline 2: all-MiniLM-L6-v2
print("="*70)
print("BASELINE 2: all-MiniLM-L6-v2")
print("="*70 + "\n")
mini_results = evaluate_model_on_dataset(model_mini, test_data, "all-MiniLM-L6-v2")
print("all-MiniLM-L6-v2 Results:")
for metric in sorted(mini_results.keys()):
    print(f"  {metric}: {mini_results[metric]:.4f}")

## Task [B2]
Fine-tune DistilBERT using supervised triplet loss to improve semantic search performance. 

Fine-tuning strategy:
1) Create triplet dataset: (query, positive_chunk, negative_chunk)
2) Train using triplet loss to minimize distance between query and correct answer
3) Maximize distance between query and random other chunks
4) Evaluate on test set and compare against baseline

In [ ]:
# Create triplet dataset for fine-tuning
import random
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import TripletMarginLoss

print("="*70)
print("PREPARING TRIPLET DATASET")
print("="*70 + "\n")

class TripletDataset:
    """Dataset that generates triplets: (anchor, positive, negative)"""
    def __init__(self, dataset):
        self.triplets = []
        for example in dataset:
            query = example['query']
            candidates = example['candidate_chunks']
            answer_pos = example['answer_pos']
            positive_chunk = candidates[answer_pos]
            negative_positions = [i for i in range(len(candidates)) if i != answer_pos]
            negative_pos = random.choice(negative_positions) # Randomly select one negative chunk
            negative_chunk = candidates[negative_pos]
            self.triplets.append((query, positive_chunk, negative_chunk))
        
        print(f"Created {len(self.triplets)} triplets from {len(dataset)} samples")
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        return self.triplets[idx]

def triplet_collate_fn(batch):
    """Extract text triplets from batch"""
    queries = [item[0] for item in batch]
    positives = [item[1] for item in batch]
    negatives = [item[2] for item in batch]
    return queries, positives, negatives

# Prepare triplets data from train dataset
triplet_dataset = TripletDataset(train_data)
train_dataloader = DataLoader(triplet_dataset, shuffle=True, batch_size=32, collate_fn=triplet_collate_fn)
print(f"DataLoader created: {len(train_dataloader)} batches\n")

In [ ]:
# Train the model with Triplet Loss
from sentence_transformers import losses, InputExample

print("="*70)
print("FINE-TUNING WITH TRIPLET LOSS")
print("="*70 + "\n")

# Create a copy of the model for fine-tuning
from copy import deepcopy
finetuned_model = deepcopy(model_bert)

# Convert triplet data to InputExample format for SentenceTransformer
train_examples = []
for anchor, positive, negative in triplet_dataset.triplets:
    train_examples.append(InputExample(texts=[anchor, positive, negative], label=0))

print(f"Created {len(train_examples)} training examples")

# Define collate function for InputExample objects
def input_example_collate_fn(batch):
    """Collate function for InputExample objects - returns batch as-is for sentence_transformers"""
    return batch

# Create new DataLoader with InputExample objects
st_train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16, collate_fn=input_example_collate_fn)
train_loss = losses.TripletLoss(model=finetuned_model, triplet_margin=0.5)

print(f"Starting fine-tuning...")
print(f"  - Loss: TripletLoss (margin=0.5)")
print(f"  - Epochs: 3")
print(f"  - Batch size: 16")
print(f"  - Total steps: {len(st_train_dataloader) * 3}\n")

# Fine-tune the model
warmup_steps = int(len(st_train_dataloader) * 3 * 0.1)

print(f"Training configuration:")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Total batches per epoch: {len(st_train_dataloader)}")
print(f"  Total training steps: {len(st_train_dataloader) * 3}\n")

finetuned_model.fit(
    train_objectives=[(st_train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=warmup_steps,
    show_progress_bar=True
)

print(f"\nFine-tuning completed")

In [ ]:
# Evaluate fine-tuned model on the same test data
print("="*70)
print("EVALUATING FINE-TUNED MODEL")
print("="*70 + "\n")

finetuned_evaluation = evaluate_model_on_dataset(finetuned_model, test_data, "Fine-tuned DistilBERT")
print("Fine-tuned Model Results (Hit and MRR):")
for metric in sorted(finetuned_evaluation.keys()):
    print(f"  {metric}: {finetuned_evaluation[metric]:.4f}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# ============== INFONCE Configuration ==============
INFONCE_CONFIG = {
    'batch_size': 16,
    'epochs': 5,
    'temperature': 0.07,
    'logit_margin': 0.05,
    'hinge_margin': 0.20,
    'hinge_weight': 0.50,
    'hard_negatives': 5,
    'train_samples': min(4000, len(train_data))
}

# Pre-compute embeddings to avoid recomputation
print("="*70)
print("PRE-COMPUTING EMBEDDINGS FOR HARD NEGATIVE MINING")
print("="*70 + "\n")

print(f"Computing embeddings for {INFONCE_CONFIG['train_samples']} training samples...")
train_samples_subset = train_data.select(range(INFONCE_CONFIG['train_samples']))

# Cache embeddings: {idx: {'query': emb, 'candidates': [embs...]}}
embeddings_cache = {}
for i, example in enumerate(train_samples_subset):
    query_emb = model_bert.encode(example['query'], convert_to_numpy=True, normalize_embeddings=True)
    candidates_embs = model_bert.encode(example['candidate_chunks'], convert_to_numpy=True, normalize_embeddings=True)
    embeddings_cache[i] = {'query': query_emb, 'candidates': candidates_embs}
    
    if (i + 1) % max(1, INFONCE_CONFIG['train_samples'] // 5) == 0:
        print(f"  Progress: {(i + 1) / INFONCE_CONFIG['train_samples'] * 100:.1f}%")

print(f"Pre-computed {len(embeddings_cache)} embedding sets\n")

# Hard Negative Dataset Class (uses pre-computed embeddings)
class HardNegativeDataset:
    """Dataset that mines hard negatives using pre-computed embeddings"""
    
    def __init__(self, dataset, embeddings_cache, num_negatives=5, num_samples=None):
        self.dataset = dataset
        self.embeddings_cache = embeddings_cache
        self.num_negatives = num_negatives
        self.num_samples = min(num_samples or len(dataset), len(embeddings_cache))
        self.examples = self._build_examples()
    
    def _build_examples(self):
        """Build training examples with hard negatives using pre-computed embeddings"""
        examples = []
        print(f"Mining hard negatives using pre-computed embeddings...")
        
        for i in range(self.num_samples):
            example = self.dataset[i]
            query, candidates, answer_pos = example['query'], example['candidate_chunks'], example['answer_pos']
            positive = candidates[answer_pos]
            
            # Get pre-computed embeddings (no recomputation!)
            query_emb = self.embeddings_cache[i]['query']
            candidates_embs = self.embeddings_cache[i]['candidates']
            
            # Rank by similarity
            sims = cosine_similarity([query_emb], candidates_embs)[0]
            sorted_idx = np.argsort(sims)[::-1]
            
            # Collect hard negatives (exclude positive)
            hard_negs = [candidates[idx] for idx in sorted_idx if idx != answer_pos][:self.num_negatives]
            
            if not hard_negs:
                continue
            
            # Pad if needed
            while len(hard_negs) < self.num_negatives:
                hard_negs.append(hard_negs[-1])
            
            examples.append(InputExample(texts=[query, positive] + hard_negs[:self.num_negatives]))
            
            if (i + 1) % max(1, self.num_samples // 5) == 0:
                print(f"  Progress: {(i + 1) / self.num_samples * 100:.1f}%")
        
        print(f"Created {len(examples)} training examples with hard negatives\n")
        return examples
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.examples[idx]

# Build hard negative dataset using cached embeddings
hard_neg_dataset = HardNegativeDataset(
    dataset=train_data,
    embeddings_cache=embeddings_cache,
    num_negatives=INFONCE_CONFIG['hard_negatives'],
    num_samples=INFONCE_CONFIG['train_samples']
)
infonce_train_examples = hard_neg_dataset.examples

In [ ]:
# InfoNCE Hard-Margin Loss (Simplified)
class InfoNCEHardMarginLoss(nn.Module):
    """InfoNCE loss with in-batch positives and mined hard negatives + margin penalty."""
    
    def __init__(self, model, temperature=0.07, logit_margin=0.05, hinge_margin=0.2, hinge_weight=0.5):
        super().__init__()
        self.model = model
        self.temperature = temperature
        self.logit_margin = logit_margin
        self.hinge_margin = hinge_margin
        self.hinge_weight = hinge_weight

    def forward(self, sentence_features, labels):
        # Encode all sentences: anchor, positive, hard negatives
        embeddings = [self.model(feat)['sentence_embedding'] for feat in sentence_features]
        anchor_emb = F.normalize(embeddings[0], p=2, dim=1)
        positive_emb = F.normalize(embeddings[1], p=2, dim=1)
        hard_neg_embs = [F.normalize(emb, p=2, dim=1) for emb in embeddings[2:]]

        # Compute logits: positive vs hard negatives (with margin boost)
        positive_logits = (anchor_emb @ positive_emb.T).diag().unsqueeze(1)
        hard_neg_logits = torch.cat([anchor_emb @ neg.T for neg in hard_neg_embs], dim=1)
        
        # Apply logit margin to hard negatives and scale
        logits = torch.cat([positive_logits, hard_neg_logits + self.logit_margin], dim=1) / self.temperature

        # Cross-entropy loss: target is positive (index 0)
        targets = torch.zeros(anchor_emb.size(0), dtype=torch.long, device=anchor_emb.device)
        contrastive_loss = F.cross_entropy(logits, targets)

        # Hinge loss: penalize hardest negative too close to positive
        hardest_neg_score = hard_neg_logits.max(dim=1).values
        margin_loss = F.relu(self.hinge_margin + hardest_neg_score - positive_logits.squeeze()).mean()

        return contrastive_loss + self.hinge_weight * margin_loss

In [ ]:
# Train the model with InfoNCE Loss + Hard Negatives
print("="*70)
print("FINE-TUNING WITH INFO-NCE LOSS + HARD NEGATIVES")
print("="*70 + "\n")

# Setup training
contrastive_model = deepcopy(model_bert)

# Define collate function for InputExample objects
def input_example_collate_fn(batch):
    """Collate function for InputExample objects - returns batch as-is for sentence_transformers"""
    return batch

infonce_dataloader = DataLoader(
    infonce_train_examples,
    shuffle=True,
    batch_size=INFONCE_CONFIG['batch_size'],
    collate_fn=input_example_collate_fn,
)
infonce_loss = InfoNCEHardMarginLoss(
    model=contrastive_model,
    temperature=INFONCE_CONFIG['temperature'],
    logit_margin=INFONCE_CONFIG['logit_margin'],
    hinge_margin=INFONCE_CONFIG['hinge_margin'],
    hinge_weight=INFONCE_CONFIG['hinge_weight'],
)

# Print configuration
print(f"Examples: {len(infonce_train_examples)} | Batches: {len(infonce_dataloader)} | "
      f"Batch size: {INFONCE_CONFIG['batch_size']} | Hard negatives: {INFONCE_CONFIG['hard_negatives']}")
print(f"Loss: temperature={INFONCE_CONFIG['temperature']}, logit_margin={INFONCE_CONFIG['logit_margin']}, "
      f"hinge_margin={INFONCE_CONFIG['hinge_margin']}, hinge_weight={INFONCE_CONFIG['hinge_weight']}\n")

# Train
contrastive_model.fit(
    train_objectives=[(infonce_dataloader, infonce_loss)],
    epochs=INFONCE_CONFIG['epochs'],
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    checkpoint_path="./distilbert-semantic-search-infonce-hard-margin",
    checkpoint_save_steps=len(infonce_dataloader),
    checkpoint_save_total_limit=2,
)

# Evaluate
infonce_evaluation = evaluate_model_on_dataset(contrastive_model, test_data, 'Info-NCE DistilBERT')

In [ ]:
# Compare Fine-tuned Model vs DistilBERT Baseline vs MiniLM Baseline
print("="*70)
print("RESULTS COMPARISON: Hit@k and MRR@k Metrics")
print("="*70 + "\n")

comparison_data = {
    'Metric': [],
    'DistilBERT (Baseline 1)': [],
    'MiniLM (Baseline 2)': [],
    'Triplets DistilBERT': [],
    'INFONCE DistilBERT': [],
    'Improvement vs DistilBERT (%)': [],
    'Improvement vs MiniLM (%)': [],
    'Improvement INFONCE vs DistilBERT (%)': [],
    'Improvement INFONCE vs MiniLM (%)': []
}

# Compare Hit@k metrics
print("HIT@K METRICS (Recall-based):")
print("-" * 70)
for metric in ['Hit@1', 'Hit@3', 'Hit@5']:
    baseline_val = bert_results[metric]
    baseline2_val = mini_results[metric]
    finetuned_val = finetuned_evaluation[metric]
    infonce_val = infonce_evaluation[metric]
    improvement_vs_1 = ((finetuned_val - baseline_val) / baseline_val) * 100
    improvement_vs_2 = ((finetuned_val - baseline2_val) / baseline2_val) * 100
    improvement_vs_3 = ((infonce_val - baseline_val) / baseline_val) * 100
    improvement_vs_4 = ((infonce_val - baseline2_val) / baseline2_val) * 100
    comparison_data['Metric'].append(metric)
    comparison_data['DistilBERT (Baseline 1)'].append(baseline_val)
    comparison_data['MiniLM (Baseline 2)'].append(baseline2_val)
    comparison_data['Triplets DistilBERT'].append(finetuned_val)
    comparison_data['INFONCE DistilBERT'].append(infonce_val)
    comparison_data['Improvement vs DistilBERT (%)'].append(improvement_vs_1)
    comparison_data['Improvement vs MiniLM (%)'].append(improvement_vs_2)
    comparison_data['Improvement INFONCE vs DistilBERT (%)'].append(improvement_vs_3)
    comparison_data['Improvement INFONCE vs MiniLM (%)'].append(improvement_vs_4)


    print(f"{metric}:")
    print(f"  DistilBERT (Baseline):     {baseline_val:.4f}")
    print(f"  MiniLM (Baseline):         {baseline2_val:.4f}")
    print(f"  Triplets DistilBERT:   {finetuned_val:.4f}")
    print(f"  INFONCE DistilBERT:   {infonce_val:.4f}")
    print(f"  Improvement vs DistilBERT: {improvement_vs_1:+.2f}%")
    print(f"  Improvement vs MiniLM:     {improvement_vs_2:+.2f}%")
    print(f"  Improvement INFONCE vs DistilBERT: {improvement_vs_3:+.2f}%")
    print(f"  Improvement INFONCE vs MiniLM:     {improvement_vs_4:+.2f}%\n")

# Compare MRR@k metrics
print("MRR@K METRICS (Ranking quality):")
print("-" * 70)
for metric in ['MRR@1', 'MRR@3', 'MRR@5']:
    baseline_val = bert_results[metric]
    baseline2_val = mini_results[metric]
    finetuned_val = finetuned_evaluation[metric]
    infonce_val = infonce_evaluation[metric]

    improvement_vs_1 = ((finetuned_val - baseline_val) / baseline_val) * 100 if baseline_val > 0 else 0
    improvement_vs_2 = ((finetuned_val - baseline2_val) / baseline2_val) * 100 if baseline2_val > 0 else 0
    improvement_vs_3 = ((infonce_val - baseline_val) / baseline_val) * 100 if baseline_val > 0 else 0
    improvement_vs_4 = ((infonce_val - baseline2_val) / baseline2_val) * 100 if baseline2_val > 0 else 0
    print(f"{metric}:")
    print(f"  DistilBERT (Baseline):     {baseline_val:.4f}")
    print(f"  MiniLM (Baseline):         {baseline2_val:.4f}")
    print(f"  Triplets DistilBERT:   {finetuned_val:.4f}")
    print(f"  INFONCE DistilBERT:   {infonce_val:.4f}")
    print(f"  Improvement vs DistilBERT: {improvement_vs_1:+.2f}%")
    print(f"  Improvement vs MiniLM:     {improvement_vs_2:+.2f}%")
    print(f"  Improvement INFONCE vs DistilBERT: {improvement_vs_3:+.2f}%")
    print(f"  Improvement INFONCE vs MiniLM:     {improvement_vs_4:+.2f}%\n")

In [ ]:
# Visualize Comparison Results: Three-Way Comparison
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Data preparation
metrics = ['Hit@1', 'Hit@3', 'Hit@5']
distilbert_scores = [comparison_data['DistilBERT (Baseline 1)'][i] for i in range(3)]
minilm_scores = [comparison_data['MiniLM (Baseline 2)'][i] for i in range(3)]
finetuned_scores = [comparison_data['Triplets DistilBERT'][i] for i in range(3)]
infonce_scores = [comparison_data['INFONCE DistilBERT'][i] for i in range(3)]

# Plot 1: Three-way side-by-side comparison
x = np.arange(len(metrics))
width = 0.25

axes[0, 0].bar(x - width, distilbert_scores, width, label='DistilBERT (Baseline)', color='steelblue', alpha=0.8)
axes[0, 0].bar(x, minilm_scores, width, label='MiniLM (Baseline)', color='skyblue', alpha=0.8)
axes[0, 0].bar(x + width, finetuned_scores, width, label='Triplets DistilBERT', color='coral', alpha=0.8)
axes[0, 0].bar(x + 2 * width, infonce_scores, width, label='INFONCE DistilBERT', color='lightgreen', alpha=0.8)
axes[0, 0].set_ylabel('Hit@k Score', fontsize=12)
axes[0, 0].set_title('Models Comparison', fontweight='bold', fontsize=13)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics)
axes[0, 0].legend(fontsize=10)
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(axis='y', alpha=0.3)
for i, (d, m, f, infonce) in enumerate(zip(distilbert_scores, minilm_scores, finetuned_scores, infonce_scores)):
    axes[0, 0].text(i - width, d + 0.02, f'{d:.3f}', ha='center', fontsize=8)
    axes[0, 0].text(i, m + 0.02, f'{m:.3f}', ha='center', fontsize=8)
    axes[0, 0].text(i + width, f + 0.02, f'{f:.3f}', ha='center', fontsize=8)
    axes[0, 0].text(i + 2 * width, infonce + 0.02, f'{infonce:.3f}', ha='center', fontsize=8)

# Plot 2: Improvement vs DistilBERT
improvements_vs_db = comparison_data['Improvement vs DistilBERT (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_vs_db]
axes[0, 1].bar(metrics, improvements_vs_db, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[0, 1].set_ylabel('Improvement (%)', fontsize=12)
axes[0, 1].set_title('Triplets DistilBERT vs DistilBERT Baseline', fontweight='bold', fontsize=13)
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[0, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_vs_db):
    axes[0, 1].text(i, v + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

# Plot 3: Improvement vs MiniLM
improvements_vs_ml = comparison_data['Improvement vs MiniLM (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_vs_ml]
axes[1, 0].bar(metrics, improvements_vs_ml, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 0].set_ylabel('Improvement (%)', fontsize=12)
axes[1, 0].set_title('Triplets DistilBERT vs MiniLM Baseline', fontweight='bold', fontsize=13)
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 0].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_vs_ml):
    axes[1, 0].text(i, v + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

# Plot 4: Improvement vs DistilBERT for INFONCE model
improvements_infonce_vs_db = comparison_data['Improvement INFONCE vs DistilBERT (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_infonce_vs_db]
axes[1, 1].bar(metrics, improvements_infonce_vs_db, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
axes[1, 1].set_ylabel('Improvement (%)', fontsize=12)
axes[1, 1].set_title('INFONCE DistilBERT vs DistilBERT Baseline', fontweight='bold', fontsize=13)
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_infonce_vs_db):
    axes[1, 1].text(i, v + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

#Plot 5: iMPROVEMENT vs MiniLM for INFONCE model
improvements_infonce_vs_ml = comparison_data['Improvement INFONCE vs MiniLM (%)']
colors = ['green' if x > 0 else 'red' for x in improvements_infonce_vs_ml]
axes[1, 1].bar(metrics, improvements_infonce_vs_ml, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5, bottom=improvements_infonce_vs_db)
axes[1, 1].set_title('INFONCE DistilBERT vs MiniLM Baseline', fontweight='bold', fontsize=13)
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 1].grid(axis='y', alpha=0.3)
for i, v in enumerate(improvements_infonce_vs_ml):
    total_improvement = improvements_infonce_vs_db[i] + v
    axes[1, 1].text(i, total_improvement + 1, f'{v:+.2f}%', ha='center', fontweight='bold', fontsize=10)

# Plot 6: Summary table
summary_text = "DETAILED COMPARISON SUMMARY\n" + "="*60 + "\n\n"
summary_text += f"Models:      DistilBERT, MiniLM-L6-v2, Triplets DistilBERT, INFONCE DistilBERT\n"
summary_text += f"Metrics:     Hit@k (Recall) + MRR@k (Ranking)\n"
summary_text += f"Fine-tuning 1: DistilBERT + TripletLoss (3 epochs)\n"
summary_text += f"Loss1:          TripletMarginLoss (margin=0.5)\n"
summary_text += f"Fine-tuning 2: INFONCE DistilBERT + INFONCELoss (3 epochs)\n"
summary_text += f"Loss 2:          InfoNCEHardMarginLoss (temperature=0.07, logit_margin=0.05, hinge_margin=0.20, hinge_weight=0.50)\n"
summary_text += f"Test Dataset:  {len(test_data)} samples\n"
summary_text += f"Training Data: {len(train_data)} triplets\n"
summary_text += "="*60 + "\n\n"

# HIT@K SECTION
summary_text += "HIT@K RESULTS:\n"
hit_k_keys = ['Hit@1', 'Hit@3', 'Hit@5']
for i, metric in enumerate(hit_k_keys):
    summary_text += f"  {metric}:\n"
    summary_text += f"    DistilBERT:  {comparison_data['DistilBERT (Baseline 1)'][i]:.4f}\n"
    summary_text += f"    MiniLM:      {comparison_data['MiniLM (Baseline 2)'][i]:.4f}\n"
    summary_text += f"    Triplets DistilBERT:  {comparison_data['Triplets DistilBERT'][i]:.4f}\n"
    summary_text += f"    Δ vs DB:     {comparison_data['Improvement vs DistilBERT (%)'][i]:+.2f}%\n"
    summary_text += f"    Δ vs ML:     {comparison_data['Improvement vs MiniLM (%)'][i]:+.2f}%\n\n"
    summary_text += f"    INFONCE DistilBERT:  {comparison_data['INFONCE DistilBERT'][i]:.4f}\n"
    summary_text += f"    Δ INFONCE vs DB:     {comparison_data['Improvement INFONCE vs DistilBERT (%)'][i]:+.2f}%\n\n"

# MRR@K SECTION
summary_text += "MRR@K RESULTS:\n"
for metric in ['MRR@1', 'MRR@3', 'MRR@5']:
    db_val = bert_results[metric]
    ml_val = mini_results[metric]
    ft_val = finetuned_evaluation[metric]
    summary_text += f"  {metric}:\n"
    summary_text += f"    DistilBERT:  {db_val:.4f}\n"
    summary_text += f"    MiniLM:      {ml_val:.4f}\n"
    summary_text += f"    Triplets DistilBERT:  {ft_val:.4f}\n\n"
    summary_text += f"    INFONCE DistilBERT:  {infonce_evaluation[metric]:.4f}\n\n"

axes[1, 1].text(0.02, 0.95, summary_text, transform=axes[1, 1].transAxes, 
                fontsize=8, verticalalignment='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8, edgecolor='gray'))
axes[1, 1].axis('off')


plt.tight_layout()
plt.show()